# 🩺 Diabetes Risk Predictor: an MLOps demo

Building a model is only half the job. **MLOps** (machine learning operations) is everything that keeps a model useful
in the real world: checking the data, choosing and tracking models, serving predictions to people, and noticing when
the world changes so the model can be updated.

This notebook walks through that whole loop on a real project, live, in about 9 minutes.

**How to read this notebook**
- **Talking points** cells explain each idea in plain language. They double as notes you can review later.
- Lines starting with 🎤 are cues for the presenter (timing and when to switch windows).
- Every step runs the project's **real** code. If a live step fails, the notebook shows a clearly labelled result
  saved from a rehearsal, so the story can continue.

> ⚕️ Educational project, not a medical tool. The predictions are estimates based on survey data, not a diagnosis.

## 0. Before the audience arrives *(presenter only, not presented)*

🎤 **Checklist**, about 5 minutes before you start:

1. **Terminal 1:** `docker compose up --build` (the API and web app)
2. **Terminal 2:** `uv run prefect server start` (the automation server)
3. **Terminal 3:** `uv run mlflow ui --backend-store-uri sqlite:///mlflow.db` (the model registry)
4. **Browser tabs:** web app http://localhost:8501 · MLflow http://127.0.0.1:5000 · Prefect http://127.0.0.1:4200
5. Fill in the settings below, then run every cell in this section. All checks should show ✅.

**First time?** Rehearse once with `MODE = "record"`: it runs everything live and saves the results as the fallback.

In [ ]:
# Make the project folder the working directory, so the project's code can be imported.
# (VS Code may start notebooks inside the demo/ folder.)
import os
import sys

if os.path.basename(os.getcwd()) == "demo":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
from demo import common
from demo import mlops_helpers as demo

# ---- Settings ---------------------------------------------------------------
# "live"   run for real; if a step fails, show the saved rehearsal result
# "record" rehearsal: run for real AND save results as the fallback
# "saved"  run nothing live; show saved results only (safest)
MODE = "live"

API_URL = "http://localhost:8000"  # local API (docker compose)
UI_URL = "http://localhost:8501"  # local web app
PREFECT_API_URL = "http://127.0.0.1:4200/api"  # local Prefect server
DEPLOYED_API_URL = "https://<your-service>.onrender.com"  # your Render URL
DEPLOYED_UI_URL = "https://<your-app>.streamlit.app"  # your Streamlit Cloud URL

common.configure(PREFECT_API_URL)  # quiet logs, use the running Prefect server
common.wake_deployed_api(DEPLOYED_API_URL)  # free tier sleeps: start waking it now
runner = common.DemoRunner(MODE, demo.FALLBACK_DIR)  # saved results live in demo/fallback/mlops

In [ ]:
# Return to a clean starting point: clears simulated users, logs and reports,
# and restores the baseline model. Never touches downloaded data or MLflow history.
demo.reset_demo()

In [ ]:
# Is everything ready? Each ❌ shows how to fix it.
demo.preflight(API_URL, PREFECT_API_URL)

## 1. The problem and the big picture

**Talking points**
- We want to estimate a person's **risk of diabetes or prediabetes** from simple questions: blood pressure, BMI,
  exercise, age and so on. Many people with prediabetes don't know they have it, so an early nudge to get tested can help.
- The model learns from a large U.S. health survey (the CDC's BRFSS).
- But a model on a laptop helps nobody. It has to reach people, keep working, and be **watched**, because the world
  changes. That full cycle is MLOps:

```
  📥 Data ─▶ ✅ Validate ─▶ 🏁 Train & compare ─▶ 📦 Registry
                                                       │
  🔁 Retrain ◀─ 🚨 Drift? ◀─ 👀 Monitor ◀─ 🌐 API & web app ◀┘
```

We'll walk around this loop once, live.

🎤 ⏱ 1 min · no code to run here

## 2. Data and validation

**Talking points**
- Our data: answers from about a quarter of a million people, each with 21 health and lifestyle questions, plus
  whether they have diabetes or prediabetes.
- "Garbage in, garbage out": a model trained on bad data quietly makes bad predictions. So the pipeline **checks
  the data first**, against a written contract of which values are allowed.

🎤 ⏱ 1 min

In [ ]:
# Load the dataset and summarize it
overview = runner.run("data_overview", demo.data_overview)
print(f"{overview['people']:,} people · {overview['features']} questions each")
print(f"{overview['diabetes_rate']:.0%} have diabetes or prediabetes")

In [ ]:
# Same data twice: once clean, once with one impossible value (a BMI of 500)
check = runner.run("validation", demo.validation_example)
print("Clean data:  ", check["clean_errors"] or "no problems ✅")
print("BMI of 500:  ", check["broken_errors"])

## 3. Choosing a model

**Talking points**
- We don't guess which algorithm is best: we **race six of them**, from very simple to very flexible.
- Each model studies part of the data and is then tested on data it hasn't seen, several times over
  (**cross-validation**). Think of practice questions vs. the real exam.
- The score is **ROC-AUC**: how well the model ranks people from lower to higher risk. 0.5 is a coin flip, 1.0 is perfect.
- Comparing the two scores reveals two classic problems:
  - **Overfitting** (high variance): great on practice data, poor on new data. The model memorized instead of learning.
  - **Underfitting** (high bias): mediocre everywhere. The model is too simple for the pattern.

🎤 ⏱ 2 min · takes ~10 seconds to run; introduce the six models while it runs

In [ ]:
# Race all six models with cross-validation (the same function the pipeline uses)
race = runner.run("compare_models", demo.compare_models)
common.show_table(
    race["table"],
    {
        "model": "Model",
        "train_score": "Score on practice data",
        "val_score": "Score on new data",
        "gap": "Gap",
        "diagnosis": "Diagnosis",
    },
)

**What to notice:** the deep decision tree scores (almost) perfectly on practice data but worst on new data: a textbook
case of overfitting. We choose the winner by its score on **new** data, never on practice data.

The **learning curve** below shows how the winner improves as it sees more data. If the two lines are close and flat,
more data won't help much; a big gap that shrinks with more data means more data would help.

In [ ]:
# The winner, and its learning curve
print(f"Winner: {race['best']}")
common.show_image(race["image_path"])

## 4. Tracking and serving

**Talking points**
- Every trained model is saved in a **model registry** (MLflow), like a library catalog: each version, its settings
  and its scores. One version carries the **production** label: the one users get.
- Users never touch the model directly. They go through an **API**, the model's front door, which also refuses
  impossible input instead of returning a meaningless answer.
- The web app is just a friendly form that calls the same API.

🎤 ⏱ 2 min · optional: show the MLflow tab (Models → diabetes-risk)

In [ ]:
# Which model is in production right now? (read from models/metadata.json)
model = demo.production_model()
print(f"Production model: version {model['version']} ({model['model_type']})")
print(f"Trained {model['trained_at']} UTC · reason: {model['reason']} · ROC-AUC {model['roc_auc']}")

In [ ]:
# Ask the running API for a prediction, then send impossible values it must refuse
answer = runner.run("prediction", lambda: demo.predict(API_URL))
result = answer["prediction"]
print(f"Risk: {result['risk_level'].upper()} ({result['probability']:.0%}) · from model v{result['model_version']}")
print(f"\nImpossible input (BMI 500, Age 40) → refused with HTTP {answer['invalid_status']}:")
for problem in answer["invalid_problems"]:
    print("  •", problem)

🎤 Switch to the **web app** tab (http://localhost:8501), fill in the form and click **Check my risk**:
same API, same model, friendlier face.

## 5. Drift and retraining

**Talking points**
- The world changes. If the people using the app start to look different from the people the model learned from,
  that's **data drift**, and the model is now working in less familiar territory.
- Monitoring compares recent users with the training data using **PSI** (Population Stability Index):
  roughly, "how much has the crowd changed?" Above 0.25 counts as significant drift.
- In real life, whether a user actually has diabetes is only known later. When those **labels arrive**, the system can
  retrain. But it's cautious: the new model (**challenger**) must pass a minimum **quality gate** AND beat the current
  one (**champion**) on recent data before it replaces it.
- All of this runs automatically on a schedule using **Prefect**, the project's automation tool. Here we trigger it by hand.

**Step A: ordinary users.** We send 1,000 simulated users through the API, then run the monitoring check.

🎤 ⏱ ~20 seconds

In [ ]:
# 1,000 ordinary users -> then the REAL Prefect monitoring flow checks for drift
normal = runner.run("normal_traffic", lambda: demo.traffic_and_monitoring(API_URL, drift=False))
demo.print_drift(normal)

**Step B: the population changes.** Now imagine the app becomes popular with an older, heavier group of people.
We send 1,000 of them, and the monitoring check runs again. If it detects drift, it **automatically starts retraining**.

🎤 ⏱ ~1 min · while it runs, switch to the Prefect tab to show the flows running live

In [ ]:
# 1,000 heavier, older users -> monitoring detects drift -> the REAL training flow runs
drifted = runner.run("drifted_traffic", lambda: demo.traffic_and_monitoring(API_URL, drift=True))
demo.print_drift(drifted)

In [ ]:
# Champion vs. challenger: did the new model earn its place?
demo.explain_retraining(drifted)

**What to notice:** either outcome is a success for MLOps.
- **New model promoted:** the system adapted to a changed world on its own, and the API switched to it without a restart.
- **Old model kept:** drift is a *warning*, not proof the model got worse. Replacing a working model without evidence
  would be reckless, so the system checked first.

## 6. Wrap-up

**Talking points**
- We went around the whole loop: data → validation → model comparison → registry → API and web app → monitoring →
  automatic retraining.
- Two things ran in the background that we didn't have time to show: **CI/CD** (GitHub automatically tests every
  change and redeploys the app) and **containers** (Docker packages the app so it runs the same everywhere).
- Next version ideas: explain *why* each person got their score, check the model is fair across groups,
  and test against real data from later years.
- The app is live on the internet:

🎤 ⏱ 30 sec · open the deployed web app in the browser

In [ ]:
# The same system, live on the internet
print("Deployed API:  ", common.deployed_status())
print("Public web app:", DEPLOYED_UI_URL)

In [ ]:
# Presenter: before committing this notebook as a handout, every step should have run live.
runner.handout_check()

## Glossary

| Term | Meaning |
|---|---|
| **MLOps** | Practices for building, deploying and maintaining ML models reliably in the real world |
| **Data validation** | Checking data against rules (allowed columns and values) before using it |
| **Cross-validation** | Testing a model several times on different held-out parts of the data |
| **ROC-AUC** | How well a model ranks cases by risk: 0.5 = coin flip, 1.0 = perfect |
| **Overfitting (high variance)** | Memorizing the training data: great on it, poor on new data |
| **Underfitting (high bias)** | Too simple to capture the pattern: mediocre everywhere |
| **Learning curve** | Chart of training vs. validation score as the amount of data grows |
| **Model registry** | A catalog of model versions with their settings, scores and labels (MLflow) |
| **API** | The model's "front door": software sends inputs, gets predictions |
| **Data drift** | The data a model sees in use changes compared with its training data |
| **PSI** | Population Stability Index: how much a feature's distribution has shifted |
| **Champion / challenger** | The current production model vs. a new candidate; the challenger must prove it's better |
| **Orchestration** | Running pipeline steps automatically, in order, on a schedule (Prefect) |
| **CI/CD** | Automatically testing every change (CI) and releasing it (CD) |
| **Container** | A packaged app with everything it needs, running the same anywhere (Docker) |

🎤 **After the demo:** run `uv run python -m demo.mlops_helpers reset` in a terminal, so `models/` matches the committed version again.

**Next:** the DevOps side of this project (how changes are checked, shipped and kept healthy) is in `demo/devops_demo.ipynb`.